# Exploratory Data Analysis — Syndicated Loan Spread Dataset

This notebook explores the merged panel produced by `src/merge.py` + `src/features.py`.

Sections:
1. Load data
2. AISD distribution
3. AISD over time
4. Correlation matrix
5. AISD vs leverage scatter

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)

ROOT = Path('..').resolve()
RAW  = ROOT / 'data' / 'raw'

df = pd.read_parquet(RAW / 'features.parquet')
df['origination_date'] = pd.to_datetime(df['origination_date'])
df['year'] = df['origination_date'].dt.year

print(f'Observations : {len(df):,}')
print(f'Date range   : {df["origination_date"].min().date()}  —  {df["origination_date"].max().date()}')
print(f'Mean AISD    : {df["aisd_bps"].mean():.1f} bps')
df.describe().T.round(2)

## 1. AISD Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Histogram
ax = axes[0]
ax.hist(df['aisd_bps'], bins=60, edgecolor='white', color='steelblue', linewidth=0.4)
ax.axvline(df['aisd_bps'].mean(),   color='crimson',  ls='--', lw=1.5, label=f'Mean = {df["aisd_bps"].mean():.0f}')
ax.axvline(df['aisd_bps'].median(), color='darkorange', ls=':',  lw=1.5, label=f'Median = {df["aisd_bps"].median():.0f}')
ax.set_xlabel('All-In Drawn Spread (bps)')
ax.set_ylabel('Count')
ax.set_title('AISD Distribution')
ax.legend(framealpha=0.8)

# Log-scale histogram
ax = axes[1]
ax.hist(np.log(df['aisd_bps'].clip(lower=1)), bins=60, edgecolor='white',
        color='teal', linewidth=0.4)
ax.set_xlabel('log(AISD)')
ax.set_ylabel('Count')
ax.set_title('log(AISD) Distribution')

plt.suptitle('All-In Drawn Spread — Distributional Properties', y=1.02, fontsize=13)
plt.tight_layout()
plt.savefig(RAW / 'figures' / 'eda_aisd_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Average AISD Over Time

In [ ]:
annual = (
    df.groupby('year')['aisd_bps']
      .agg(mean='mean', median='median', q25=lambda x: x.quantile(0.25),
           q75=lambda x: x.quantile(0.75), n='count')
      .reset_index()
)

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.fill_between(annual['year'], annual['q25'], annual['q75'],
                alpha=0.18, color='steelblue', label='IQR')
ax.plot(annual['year'], annual['mean'],   color='steelblue', lw=2.2, label='Mean')
ax.plot(annual['year'], annual['median'], color='darkorange', lw=1.8,
        ls='--', label='Median')

# Annotate key episodes
ax.axvspan(2008, 2009, alpha=0.08, color='red', label='GFC')
ax.axvspan(2020, 2020.5, alpha=0.08, color='purple', label='COVID-19')

ax.set_xlabel('Year')
ax.set_ylabel('AISD (bps)')
ax.set_title('Average Syndicated Loan Spread Over Time')
ax.legend(ncol=3, framealpha=0.85)
ax.xaxis.set_major_locator(mticker.MultipleLocator(2))

# Secondary axis: deal count
ax2 = ax.twinx()
ax2.bar(annual['year'], annual['n'], alpha=0.15, color='grey', width=0.7)
ax2.set_ylabel('# Loans in sample', color='grey')
ax2.tick_params(axis='y', labelcolor='grey')

plt.tight_layout()
plt.savefig(RAW / 'figures' / 'eda_aisd_time_series.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Correlation Matrix

In [ ]:
CORR_COLS = [
    'aisd_bps', 'leverage', 'profitability', 'interest_coverage',
    'log_firm_size', 'tangibility', 'cash_ratio', 'altman_z',
    'log_facility_amount', 'maturity_months', 'collateral_flag',
    'eq_vol_12m', 'baa_aaa_spread',
]
avail = [c for c in CORR_COLS if c in df.columns]
corr  = df[avail].corr()

fig, ax = plt.subplots(figsize=(11, 9))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f',
    cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    linewidths=0.5, linecolor='white',
    annot_kws={'size': 7.5}, ax=ax,
)
ax.set_title('Pairwise Correlation Matrix', fontsize=13, pad=14)
plt.tight_layout()
plt.savefig(RAW / 'figures' / 'eda_correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. AISD vs Leverage

In [ ]:
plot_df = df[['aisd_bps', 'leverage', 'log_firm_size', 'is_revolver']].dropna()
plot_df = plot_df[(plot_df['leverage'] >= 0) & (plot_df['leverage'] <= 2)]

fig, ax = plt.subplots(figsize=(9, 5))
scatter = ax.scatter(
    plot_df['leverage'],
    plot_df['aisd_bps'],
    c=plot_df['log_firm_size'],
    cmap='viridis_r',
    alpha=0.35,
    s=14,
    linewidths=0,
)
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('log(Total Assets)', fontsize=9)

# LOWESS trend line
from statsmodels.nonparametric.smoothers_lowess import lowess
lev_sorted = plot_df['leverage'].values
aisd_sorted = plot_df['aisd_bps'].values
smoothed = lowess(aisd_sorted, lev_sorted, frac=0.2, return_sorted=True)
ax.plot(smoothed[:, 0], smoothed[:, 1], color='crimson', lw=2.2, label='LOWESS')

ax.set_xlabel('Leverage (Total Debt / Total Assets)')
ax.set_ylabel('AISD (bps)')
ax.set_title('AISD vs Leverage (coloured by firm size)')
ax.legend()
plt.tight_layout()
plt.savefig(RAW / 'figures' / 'eda_aisd_vs_leverage.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Key Descriptive Statistics by Loan Type

In [ ]:
if 'is_revolver' in df.columns:
    tbl = (
        df.groupby('is_revolver')[['aisd_bps', 'leverage', 'log_firm_size',
                                    'maturity_months', 'log_facility_amount']]
          .agg(['mean', 'median', 'std'])
          .round(2)
    )
    tbl.index = tbl.index.map({0: 'Term Loan', 1: 'Revolver'})
    display(tbl)
else:
    print('is_revolver column not found')